# Day 13: NLP Text Preprocessing and TF-IDF

**Objective:** Learn basic Natural Language Processing (NLP), lowercase text, tokenize, remove stopwords, and convert text into TF-IDF features.

This small sample dataset is for practice and demonstrates sentiment-labelled text. It is not a production model.

In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from IPython.display import display


## 1. Create a small text dataset

In [2]:
data = {
    'text': [
        'I love this product, it works really well!',
        'The service was excellent and the staff were helpful.',
        'Amazing quality and fast delivery.',
        'This is a great experience, I am very happy.',
        'The app is easy to use and useful.',
        'I am satisfied with the results.',
        'The product is terrible and stopped working.',
        'Very poor service and rude staff.',
        'The delivery was late and the package was damaged.',
        'I dislike this app because it is slow.',
        'Bad quality, not worth the money.',
        'I am unhappy with this disappointing experience.',
        'The product is okay but could be better.',
        'Delivery arrived today, nothing special.',
        'The app has basic features and a simple design.',
        'It works as described, with average performance.'
    ],
    'sentiment': ['positive']*6 + ['negative']*6 + ['neutral']*4
}
df = pd.DataFrame(data)
display(df.head())
print('Dataset shape:', df.shape)

,text,sentiment
0,"I love this product, it works really well!",positive
1,The service was excellent and the staff were h...,positive
2,Amazing quality and fast delivery.,positive
3,"This is a great experience, I am very happy.",positive
4,The app is easy to use and useful.,positive


Dataset shape: (16, 2)


## 2. Lowercasing
Lowercasing converts uppercase letters to lowercase so that words such as `Great` and `great` are treated consistently.

In [3]:
df['lowercase_text'] = df['text'].str.lower()
display(df[['text', 'lowercase_text']].head())

,text,lowercase_text
0,"I love this product, it works really well!","i love this product, it works really well!"
1,The service was excellent and the staff were h...,the service was excellent and the staff were h...
2,Amazing quality and fast delivery.,amazing quality and fast delivery.
3,"This is a great experience, I am very happy.","this is a great experience, i am very happy."
4,The app is easy to use and useful.,the app is easy to use and useful.


## 3. Tokenization
Tokenization splits a text string into smaller units called tokens. This simple example removes punctuation and splits on whitespace.

In [4]:
def tokenize(text):
    cleaned = re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower())
    return cleaned.split()

df['tokens'] = df['text'].apply(tokenize)
display(df[['text', 'tokens']].head())

,text,tokens
0,"I love this product, it works really well!","[i, love, this, product, it, works, really, well]"
1,The service was excellent and the staff were h...,"[the, service, was, excellent, and, the, staff..."
2,Amazing quality and fast delivery.,"[amazing, quality, and, fast, delivery]"
3,"This is a great experience, I am very happy.","[this, is, a, great, experience, i, am, very, ..."
4,The app is easy to use and useful.,"[the, app, is, easy, to, use, and, useful]"


## 4. Stopword removal
Stopwords are common words such as `the`, `is`, and `and`. Removing them can reduce noise for some tasks, although stopwords should not be removed automatically for every NLP problem.

In [5]:
stop_words = set(ENGLISH_STOP_WORDS)
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df['tokens_no_stopwords'] = df['tokens'].apply(remove_stopwords)
df['clean_text'] = df['tokens_no_stopwords'].apply(lambda tokens: ' '.join(tokens))
display(df[['text', 'tokens', 'tokens_no_stopwords', 'clean_text']].head())

,text,tokens,tokens_no_stopwords,clean_text
0,"I love this product, it works really well!","[i, love, this, product, it, works, really, well]","[love, product, works, really]",love product works really
1,The service was excellent and the staff were h...,"[the, service, was, excellent, and, the, staff...","[service, excellent, staff, helpful]",service excellent staff helpful
2,Amazing quality and fast delivery.,"[amazing, quality, and, fast, delivery]","[amazing, quality, fast, delivery]",amazing quality fast delivery
3,"This is a great experience, I am very happy.","[this, is, a, great, experience, i, am, very, ...","[great, experience, happy]",great experience happy
4,The app is easy to use and useful.,"[the, app, is, easy, to, use, and, useful]","[app, easy, use, useful]",app easy use useful


## 5. TF-IDF vectorization
TF-IDF (Term Frequency–Inverse Document Frequency) converts text into numerical features. A term receives a higher score when it is important in a document but less common across the collection.

In [6]:
vectorizer = TfidfVectorizer(stop_words='english')
X_tfidf = vectorizer.fit_transform(df['text'])
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer.get_feature_names_out())
print('TF-IDF matrix shape:', X_tfidf.shape)
display(tfidf_df.head())

TF-IDF matrix shape: (16, 48)


,amazing,app,arrived,average,bad,basic,better,damaged,delivery,described,...,staff,stopped,terrible,today,unhappy,use,useful,working,works,worth
0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.474701,0.0
1,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.464386,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0
2,0.545086,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.424762,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0
3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0
4,0.000000,0.410292,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.526517,0.526517,0.0,0.000000,0.0


## 6. Inspect strongest average terms

In [7]:
mean_scores = tfidf_df.mean().sort_values(ascending=False)
display(mean_scores.head(12).to_frame('average_tfidf'))


,average_tfidf
product,0.082354
app,0.078496
delivery,0.077834
experience,0.065544
staff,0.058048
service,0.058048
works,0.057745
quality,0.057745
results,0.044194
satisfied,0.044194


## 7. Save outputs

In [8]:
df.to_csv('preprocessed_text_dataset.csv', index=False)
tfidf_df.to_csv('tfidf_matrix.csv', index=False)
print('Saved preprocessed_text_dataset.csv and tfidf_matrix.csv')

Saved preprocessed_text_dataset.csv and tfidf_matrix.csv


## 8. Key observations
- Lowercasing standardizes text.
- Tokenization splits sentences into words.
- Stopword removal removes selected common words.
- TF-IDF produces a sparse numerical representation of text.
- TF-IDF values are relative weights, not sentiment predictions by themselves.

**Conclusion:** This notebook demonstrates a basic text preprocessing pipeline. For a real application, evaluate preprocessing choices and use a suitable train/test split before training a classifier.